In [2]:
names = open('names.txt', 'r').read().splitlines()
names[:10]

['emma',
 'olivia',
 'ava',
 'isabella',
 'sophia',
 'charlotte',
 'mia',
 'amelia',
 'harper',
 'evelyn']

### Dataset builder

In [11]:
vocab = sorted(list(set(''.join(names))))
vocab.append('<S>')

char_to_index = {c : i for i, c in enumerate(vocab)}
index_to_char = {i : c for c, i in char_to_index.items()}

In [32]:
import torch

xs = []
ys = []
for name in names:
    chars = ['<S>'] + list(name) + ['<S>']
    for c1, c2 in zip(chars, chars[1:]):
        xs.append(char_to_index[c1])
        ys.append(char_to_index[c2])

x_train, y_train = xs[:int(len(xs) * 0.9)], ys[:int(len(ys) * 0.9)]
x_test, y_test = xs[len(x_train): ], ys[len(y_train): ]

x_train = torch.tensor(x_train)
y_train = torch.tensor(y_train)

print(f'{len(x_train)=}\n{len(x_test)=}')
len(x_train) + len(x_test) == len(xs)

len(x_train)=205331
len(x_test)=22815


True

### Training


In [34]:
from torch.nn import functional as F

W = torch.randn((len(vocab), len(vocab)), requires_grad=True)
alpha = 20

def forward(x):
    xenc = F.one_hot(x, num_classes=27).float()
    logits = xenc.matmul(W)
    return logits

def nll_loss(logits, y):
    nll = -F.log_softmax(logits, dim=1)
    nll = nll[range(len(y)), y]
    nll = nll.mean()
    return nll

for i in range(1000):
    logits = forward(x_train)
    nll = nll_loss(logits, y_train)
    loss = nll
    
    W.grad = None
    loss.backward()
    if i % 100 == 0:
        print(loss.item())
    
    W.data -= alpha * W.grad
        

3.8556532859802246
2.4985508918762207
2.467038631439209
2.4567465782165527
2.4521217346191406
2.449603796005249
2.4480206966400146
2.446927309036255
2.4461255073547363
2.445512294769287


### Inference